In [ ]:
# ============================================================================
# ETAPA 9 — MODELO DE SOBREVIVÊNCIA P→N (Pastagem → Vegetação Nativa)
# ============================================================================
# Paralelo ao etapa8 (P→S), mas para regeneração.
#
# Diferenças em relação ao etapa8:
#   1. Dataset: dataset_PN (threshold 10 anos, T_max=2014)
#   2. Class weighting: label=1 é 15% dos dados — peso 5x maior
#   3. Horizonte de avaliação: 10 anos (não 5)
#   4. Interpretação: k>1 = risco crescente de regeneração
#      (acúmulo de rebrota subterrânea com o tempo)
#
# Semelhanças com etapa8:
#   • Mesma arquitetura WeibullSurvivalModel
#   • Mesma loss (neg log-lik Weibull com censura)
#   • Mesmo clamping de parâmetros
#   • Mesmo split hexagonal (0% overlap)
#   • Mesmas 290 features
#
# Nota ecológica:
#   Regeneração no Cerrado é facilitada por estruturas lenhosas
#   subterrâneas (xilopódios) que persistem após conversão superficial.
#   Pixels com conversão incompleta têm maior probabilidade de
#   regeneração — mecanismo distinto de P→S.
# ============================================================================

import os
os.environ['KMP_DUPLICATE_LIB_OK'] = 'TRUE'

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

import pandas as pd
import numpy as np
from pathlib import Path
from tqdm.notebook import tqdm
import json
from datetime import datetime
from sklearn.metrics import roc_auc_score, precision_recall_fscore_support

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
SEED   = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

# Horizonte de avaliação — 10 anos (threshold ecológico P→N)
HORIZON_EVAL  = 10
HORIZONTE_MAX = 10   # janela máxima de observação
ANO_FIM       = 2024

# Class weight: label=1 é ~15% → peso 5x para balancear
# Calculado como: n_label0 / n_label1 = 6485/1157 ≈ 5.6
WEIGHT_LABEL1 = 5.6

print(f"✅ Configuração OK")
print(f"   Device:          {DEVICE}")
print(f"   Transição:       P→N (Pastagem → Vegetação Nativa)")
print(f"   Horizonte:       {HORIZON_EVAL} anos")
print(f"   Class weight:    {WEIGHT_LABEL1}x para label=1 (regeneração)")
print(f"   Dataset:         dataset_PN (threshold={HORIZONTE_MAX} anos, T_max=2014)")

In [ ]:
# ============================================================================
# MODELO — idêntico ao etapa8 com clamping
# ============================================================================

class WeibullSurvivalModel(nn.Module):
    def __init__(self, input_dim=290, hidden_dim=256,
                 encoder_dim=128, dropout=0.3):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, hidden_dim), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(hidden_dim, encoder_dim), nn.ReLU(), nn.Dropout(dropout)
        )
        self.weibull_head = nn.Sequential(
            nn.Linear(encoder_dim, 64), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(64, 2)
        )
        # Inicialização para regeneração:
        # k ≈ 1.0 (risco aproximadamente constante)
        # lambda ≈ 15 anos (tempo mediano longo — regeneração rara)
        with torch.no_grad():
            self.weibull_head[-1].bias[0] = np.log(1.0)
            self.weibull_head[-1].bias[1] = np.log(15.0)
        self.log_k_min   = np.log(0.1);  self.log_k_max   = np.log(5.0)
        self.log_lam_min = np.log(0.5);  self.log_lam_max = np.log(50.0)

    def forward(self, features):
        enc = self.encoder(features)
        wb  = self.weibull_head(enc)
        log_k   = wb[:, 0].clamp(self.log_k_min, self.log_k_max)
        log_lam = wb[:, 1].clamp(self.log_lam_min, self.log_lam_max)
        return log_k, log_lam

    def survival_prob(self, features, t):
        log_k, log_lam = self.forward(features)
        k = torch.exp(log_k); lam = torch.exp(log_lam)
        return torch.exp(-(t / lam) ** k)

    def regen_prob(self, features, t):
        """P(regenerar em ≤t anos) = 1 - S(t)"""
        return 1 - self.survival_prob(features, t)


# Loss com class weighting
def weibull_loss_weighted(log_k, log_lam, times, events,
                          weight_event=WEIGHT_LABEL1):
    """
    Log-verossimilhança Weibull com peso maior para eventos de regeneração.
    Compensa o desequilíbrio 85/15 entre label=0 e label=1.
    """
    eps = 1e-6
    k   = torch.exp(log_k)   + eps
    lam = torch.exp(log_lam) + eps
    t   = times.clamp(min=eps)
    ev  = events.float()

    log_f = (torch.log(k) - torch.log(lam) +
             (k-1)*(torch.log(t) - torch.log(lam)) - (t/lam)**k)
    log_S = -((t/lam)**k)

    # Peso maior para eventos (regeneração)
    weights = 1.0 + ev * (weight_event - 1.0)
    ll = weights * (ev * log_f + (1-ev) * log_S)
    return -ll.mean()


# Verificação
torch.manual_seed(SEED)
model = WeibullSurvivalModel().to(DEVICE)
total_params = sum(p.numel() for p in model.parameters())

dummy = torch.randn(4, 290).to(DEVICE)
lk, ll = model(dummy)

print(f"✅ WeibullSurvivalModel (P→N) criado!")
print(f"   Parâmetros: {total_params:,}")
print(f"   k inicial:      {torch.exp(lk).mean().item():.3f}  (esperado ≈ 1.0)")
print(f"   lambda inicial: {torch.exp(ll).mean().item():.3f}  (esperado ≈ 15.0)")
print(f"   P(≤10 anos):    {model.regen_prob(dummy, 10.0).mean().item():.3f}")
print(f"   Class weight:   {WEIGHT_LABEL1}x para eventos de regeneração")

In [ ]:
# ============================================================================
# DATASET P→N — survival com censurados
# ============================================================================

import rasterio
from rasterio.windows import Window

DATA_DIR    = r"D:\Projetos\Cerrado\LULC"
PATTERN     = "brazil_coverage_{ano}_Cerrado.tif"
YEARS       = list(range(1985, 2025))
NODATA_IN   = 255; NODATA_OUT = 0
PATCH_N     = 7;   MAX_SERIE  = len(YEARS)-1;  PATCH_YEARS = 5

def _ler_pixel(year, row, col):
    with rasterio.open(os.path.join(
            DATA_DIR, PATTERN.format(ano=year))) as ds:
        return int(ds.read(1, window=Window(col,row,1,1),
                           out_dtype="uint8")[0,0])

def _ler_patch(year, row, col):
    h = PATCH_N//2
    with rasterio.open(os.path.join(
            DATA_DIR, PATTERN.format(ano=year))) as ds:
        H,W = ds.height, ds.width
        c0 = min(max(0,col-h),W-PATCH_N)
        r0 = min(max(0,row-h),H-PATCH_N)
        arr = ds.read(1, window=Window(c0,r0,PATCH_N,PATCH_N),
                      out_dtype="uint8")
    return np.where(arr==NODATA_IN,NODATA_OUT,arr).astype(np.float32)


class SurvivalDatasetPN(Dataset):
    """
    Dataset P→N com tempo de sobrevivência.
    tempo = min(ano_regeneracao - T, HORIZONTE_MAX) se regenerou
    tempo = min(ANO_FIM - T, HORIZONTE_MAX) se não regenerou
    """

    def __init__(self, df):
        self.df = df.reset_index(drop=True)
        n_ev = self.df['evento'].sum()
        print(f"  Dataset: {len(self.df):,} samples")
        print(f"  Eventos (regenerou):  {int(n_ev):,} ({100*n_ev/len(self.df):.1f}%)")
        print(f"  Censurados:           {int(len(self.df)-n_ev):,}")

    def __len__(self): return len(self.df)

    def __getitem__(self, idx):
        r     = self.df.iloc[idx]
        row   = int(r['row']); col   = int(r['col'])
        T     = int(r['T']);   tempo = float(r['tempo'])
        evento = int(r['evento'])
        grupo = str(r.get('grupo_T', 'NC'))

        anos_s = [y for y in YEARS if y < T]
        if anos_s:
            sr = np.array([_ler_pixel(y, row, col)
                           for y in anos_s], dtype=np.float32)
            sr = np.where(sr==NODATA_IN,NODATA_OUT,sr).astype(np.float32)
        else:
            sr = np.array([], dtype=np.float32)
        sl = len(sr)
        serie = np.zeros(MAX_SERIE, dtype=np.float32)
        if sl > 0: serie[MAX_SERIE-sl:] = sr

        anos_p = [y for y in YEARS if y<T][-PATCH_YEARS:]
        patch  = np.zeros((PATCH_YEARS,PATCH_N,PATCH_N), dtype=np.float32)
        for i,y in enumerate(anos_p):
            patch[PATCH_YEARS-len(anos_p)+i] = _ler_patch(y, row, col)

        has21 = float(np.sum(sr==21)/sl) if sl>0 else 0.0
        apast = sum(1 for v in reversed(sr) if v==15) if sl>0 else 0
        ctm1  = float(sr[-1]) if sl>0 else 0.0
        aux   = np.array([has21, float(apast), ctm1], dtype=np.float32)

        isCC = 1.0 if grupo=='CC* (cluster)'  else 0.0
        isCD = 1.0 if grupo=='CD* (disperso)' else 0.0
        isNC = 1.0 if grupo=='NC'             else 0.0
        ctx  = np.array([isCC, isCD, isNC], dtype=np.float32)

        features = np.concatenate([serie, patch.flatten(), aux, ctx])
        assert features.shape == (290,)

        return (
            torch.tensor(features, dtype=torch.float32),
            torch.tensor(tempo,    dtype=torch.float32),
            torch.tensor(evento,   dtype=torch.float32)
        )


# Construir splits
BASE_DIR = Path(r"D:\Projetos\Cerrado\GeoFM_sampling")
DATA_PN  = BASE_DIR / "dataset_PN"

def preparar_split(csv_path):
    df = pd.read_csv(csv_path)
    df = df[df['label'].notna()].copy()
    df['label'] = df['label'].astype(int)
    df['T']     = df['T'].astype(int)

    # Adicionar grupo_T do dataset PN completo
    df_full = pd.read_csv(DATA_PN / "dataset_PN_full.csv")
    if 'grupo_T' in df_full.columns:
        df = df.merge(
            df_full[['row','col','T','grupo_T']].drop_duplicates(),
            on=['row','col','T'], how='left')
    df['grupo_T'] = df.get('grupo_T', pd.Series(['NC']*len(df))).fillna('NC')

    # Calcular tempo de sobrevivência
    # Para eventos: ano_regeneracao - T (se disponível)
    # Para censurados: min(ANO_FIM - T, HORIZONTE_MAX)
    df_full2 = pd.read_csv(DATA_PN / "dataset_PN_full.csv")
    if 'ano_regeneracao' in df_full2.columns:
        df = df.merge(
            df_full2[['row','col','T','ano_regeneracao']].drop_duplicates(),
            on=['row','col','T'], how='left')
    else:
        df['ano_regeneracao'] = np.nan

    df['tempo'] = np.where(
        df['label'] == 1,
        (df['ano_regeneracao'] - df['T']).clip(1, HORIZONTE_MAX),
        np.minimum(ANO_FIM - df['T'], HORIZONTE_MAX)
    )
    df['evento'] = df['label']
    df = df[df['tempo'] > 0].copy()
    return df


print("Carregando datasets P→N...")
print()
print("Train:")
df_train = preparar_split(DATA_PN / "dataset_PN_train.csv")
train_ds = SurvivalDatasetPN(df_train)
print()
print("Val:")
df_val  = preparar_split(DATA_PN / "dataset_PN_val.csv")
val_ds  = SurvivalDatasetPN(df_val)
print()
print("Test:")
df_test = preparar_split(DATA_PN / "dataset_PN_test.csv")
test_ds = SurvivalDatasetPN(df_test)

BATCH_SIZE = 256
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=0)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
print(f"\n✅ DataLoaders prontos!")

In [ ]:
# ============================================================================
# TREINO
# ============================================================================

def train_epoch(model, loader, optimizer, device):
    model.train()
    total_loss = n_batches = 0
    k_vals, lam_vals = [], []

    pbar = tqdm(loader, desc="Training")
    for features, times, events in pbar:
        features = features.to(device)
        times    = times.to(device)
        events   = events.to(device)

        log_k, log_lam = model(features)
        loss = weibull_loss_weighted(log_k, log_lam, times, events)

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        total_loss += loss.item()
        n_batches  += 1
        k_vals.extend(torch.exp(log_k).detach().cpu().numpy())
        lam_vals.extend(torch.exp(log_lam).detach().cpu().numpy())

        pbar.set_postfix({'loss': f'{loss.item():.4f}',
                          'k':    f'{np.mean(k_vals[-50:]):.3f}',
                          'lam':  f'{np.mean(lam_vals[-50:]):.1f}'})

    return (total_loss/n_batches,
            float(np.mean(k_vals)),
            float(np.mean(lam_vals)))


def evaluate(model, loader, device, horizon=HORIZON_EVAL, collect=False):
    model.eval()
    total_loss = n_batches = 0
    all_probs, all_events = [], []
    all_k, all_lam = [], []

    with torch.no_grad():
        for features, times, events in tqdm(loader, desc="Eval"):
            features = features.to(device)
            times    = times.to(device)
            events   = events.to(device)

            log_k, log_lam = model(features)
            loss = weibull_loss_weighted(log_k, log_lam, times, events)

            k   = torch.exp(log_k)
            lam = torch.exp(log_lam)
            p_regen = 1 - torch.exp(-(horizon/lam)**k)

            total_loss += loss.item()
            n_batches  += 1
            all_probs.extend(p_regen.cpu().numpy())
            all_events.extend(events.cpu().numpy())
            if collect:
                all_k.extend(k.cpu().numpy())
                all_lam.extend(lam.cpu().numpy())

    probs_np  = np.array(all_probs)
    events_np = np.array(all_events)

    try:    auc = roc_auc_score(events_np, probs_np)
    except: auc = 0.0

    preds = (probs_np >= 0.5).astype(int)
    prec, rec, f1, _ = precision_recall_fscore_support(
        events_np, preds, average='binary', zero_division=0)
    acc = (preds == events_np).mean()

    metrics = {'loss': total_loss/n_batches, 'auc': float(auc),
               'f1': float(f1), 'accuracy': float(acc),
               'precision': float(prec), 'recall': float(rec)}
    extras = (np.array(all_k), np.array(all_lam),
              probs_np, events_np) if collect else None
    return metrics, extras


# Treino
torch.manual_seed(SEED)
model     = WeibullSurvivalModel().to(DEVICE)
optimizer = optim.Adam(model.parameters(), lr=0.0005, weight_decay=1e-4)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', patience=5, factor=0.5, verbose=True)

N_EPOCHS = 30; EARLY_STOP_PAT = 10
best_val_loss = float('inf'); best_model_state = None
best_val_metrics = None; patience_counter = 0
history = {'train_loss':[], 'k_mean':[], 'lam_mean':[], 'val_metrics':[]}

print("=" * 70)
print("TREINO — WeibullSurvivalModel P→N")
print("=" * 70)
print(f"  Train: {len(train_ds):,}  Val: {len(val_ds):,}  Test: {len(test_ds):,}")
print(f"  Loss: Weibull log-lik + class weight {WEIGHT_LABEL1}x para regeneração")
print(f"  Horizonte de avaliação: {HORIZON_EVAL} anos")
print()

for epoch in range(N_EPOCHS):
    print(f"Epoch {epoch+1}/{N_EPOCHS}")
    train_loss, k_mean, lam_mean = train_epoch(
        model, train_loader, optimizer, DEVICE)
    val_metrics, _ = evaluate(
        model, val_loader, DEVICE, collect=False)
    scheduler.step(val_metrics['loss'])

    history['train_loss'].append(train_loss)
    history['k_mean'].append(k_mean)
    history['lam_mean'].append(lam_mean)
    history['val_metrics'].append(val_metrics)

    print(f"Train - Loss: {train_loss:.4f}  k={k_mean:.3f}  lam={lam_mean:.2f}")
    print(f"Val   - Loss: {val_metrics['loss']:.4f}  "
          f"AUC: {val_metrics['auc']:.4f}  F1: {val_metrics['f1']:.4f}")

    if val_metrics['loss'] < best_val_loss:
        best_val_loss    = val_metrics['loss']
        best_model_state = {k: v.cpu().clone()
                            for k, v in model.state_dict().items()}
        best_val_metrics = val_metrics.copy()
        patience_counter = 0
        print("  → Best model updated!")
    else:
        patience_counter += 1
        if patience_counter >= EARLY_STOP_PAT:
            print(f"\nEarly stopping após {epoch+1} epochs.")
            break
    print()

print(f"Melhor val AUC: {best_val_metrics['auc']:.4f}")
print(f"Melhor val F1:  {best_val_metrics['f1']:.4f}")

In [ ]:
# ============================================================================
# AVALIAÇÃO E FREEZE
# ============================================================================

import matplotlib.pyplot as plt

model.load_state_dict(best_model_state)
model.to(DEVICE)

test_metrics, (k_np, lam_np, probs_np, events_np) = evaluate(
    model, test_loader, DEVICE, collect=True)

val_test_gap = test_metrics['accuracy'] - best_val_metrics['accuracy']

print("=" * 70)
print("AVALIAÇÃO FINAL — P→N TEST SET")
print("=" * 70)
print(f"\n{'Métrica':<15} {'Validation':>12} {'Test':>12}")
print("-" * 42)
for k in ['accuracy','precision','recall','f1','auc']:
    print(f"{k:<15} {best_val_metrics[k]:>12.4f} {test_metrics[k]:>12.4f}")
print(f"\nVal-Test Gap: {val_test_gap*100:+.2f}pp")

# Parâmetros Weibull
print(f"\nParâmetros Weibull P→N por pixel (test set):")
print(f"  k:      mean={k_np.mean():.3f}  std={k_np.std():.3f}")
print(f"  lambda: mean={lam_np.mean():.2f}   std={lam_np.std():.2f}")

# Comparação P→S vs P→N
print(f"\n{'='*70}")
print("COMPARAÇÃO P→S vs P→N (Weibull)")
print(f"{'='*70}")
print(f"{'Modelo':<30} {'AUC':>7} {'k médio':>9} {'λ médio':>9}")
print("-" * 58)
print(f"{'Weibull P→S (etapa8)':<30} {'0.846':>7} {'1.881':>9} {'15.45':>9}")
print(f"{'Weibull P→N (etapa9)':<30} {test_metrics['auc']:>7.4f} "
      f"{k_np.mean():>9.3f} {lam_np.mean():>9.2f}")

# Gráfico comparativo
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

grupos = ['CC* (cluster)', 'CD* (disperso)', 'NC']
cores  = {'CC* (cluster)':'#c0392b','CD* (disperso)':'#2980b9','NC':'#27ae60'}

df_test_aug = df_test.reset_index(drop=True).copy()
n = min(len(k_np), len(df_test_aug))
df_test_aug = df_test_aug.iloc[:n].copy()
df_test_aug['k_pred']    = k_np[:n]
df_test_aug['lam_pred']  = lam_np[:n]
df_test_aug['P_regen10'] = probs_np[:n]

# k por grupo
for g in grupos:
    sub = df_test_aug[df_test_aug['grupo_T']==g]['k_pred']
    if len(sub)==0: continue
    axes[0].hist(sub, bins=25, alpha=0.6, color=cores[g], density=True,
                 label=f'{g} (k̄={sub.mean():.2f})')
axes[0].axvline(x=1, color='black', linestyle='--', alpha=0.7)
axes[0].set_title('k P→N por Grupo Ecológico', fontweight='bold')
axes[0].set_xlabel('k (forma Weibull)'); axes[0].legend(fontsize=8)

# Curvas de sobrevivência P→N por grupo
t_plot = np.linspace(0.1, 25, 200)
for g in grupos:
    sub = df_test_aug[df_test_aug['grupo_T']==g]
    if len(sub)==0: continue
    k_g = sub['k_pred'].mean(); lam_g = sub['lam_pred'].mean()
    S_g = np.exp(-(t_plot/lam_g)**k_g)
    axes[1].plot(t_plot, S_g, color=cores[g], linewidth=2,
                 label=f'{g} (k={k_g:.2f}, λ={lam_g:.1f})')
axes[1].axvline(x=10, color='gray', linestyle=':', alpha=0.7)
axes[1].set_xlabel('Anos em pastagem'); axes[1].set_ylabel('S(t)')
axes[1].set_title('Sobrevivência P→N por Grupo', fontweight='bold')
axes[1].legend(fontsize=8); axes[1].set_xlim(0,25); axes[1].set_ylim(0,1)

# Sobreposição P→N vs P→S (curvas médias)
k_ps=1.881; lam_ps=15.45   # etapa8
k_pn=k_np.mean(); lam_pn=lam_np.mean()
S_ps = np.exp(-(t_plot/lam_ps)**k_ps)
S_pn = np.exp(-(t_plot/lam_pn)**k_pn)
axes[2].plot(t_plot, 1-S_ps, color='#c0392b', linewidth=2.5, label='P→S (conversão)')
axes[2].plot(t_plot, 1-S_pn, color='#27ae60', linewidth=2.5, label='P→N (regeneração)')
axes[2].axvline(x=5,  color='gray', linestyle=':', alpha=0.5)
axes[2].axvline(x=10, color='gray', linestyle='--', alpha=0.5)
axes[2].set_xlabel('Anos em pastagem')
axes[2].set_ylabel('P(evento ≤t anos)')
axes[2].set_title('P→S vs P→N: probabilidade acumulada', fontweight='bold')
axes[2].legend(); axes[2].set_xlim(0,25); axes[2].set_ylim(0,1)

plt.suptitle('Weibull P→N — Regeneração Pastagem→Nativa',
             fontsize=12, fontweight='bold')
plt.tight_layout()

# Freeze
OUT_DIR = BASE_DIR / "etapa9_frozen"
OUT_DIR.mkdir(exist_ok=True, parents=True)
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

plt.savefig(str(OUT_DIR / f'weibull_PN_{timestamp}.png'),
            dpi=150, bbox_inches='tight')
plt.show()

checkpoint = {
    'model_state_dict': best_model_state,
    'model_class': 'WeibullSurvivalModel_PN',
    'config': {'input_dim':290,'hidden_dim':256,'encoder_dim':128,'dropout':0.3},
    'transition': 'P_to_N',
    'metrics': {
        'validation': {k: float(v) for k,v in best_val_metrics.items()},
        'test':       {k: float(v) for k,v in test_metrics.items()},
        'val_test_gap': float(val_test_gap)
    },
    'weibull_params': {
        'k_mean': float(k_np.mean()), 'k_std': float(k_np.std()),
        'lam_mean': float(lam_np.mean()), 'lam_std': float(lam_np.std())
    },
    'training': {
        'horizon_eval': HORIZON_EVAL, 'weight_label1': WEIGHT_LABEL1,
        'n_epochs': len(history['train_loss']), 'seed': SEED
    }
}
pth = OUT_DIR / f"weibull_PN_{timestamp}.pth"
torch.save(checkpoint, pth)
np.save(OUT_DIR / f"k_test_{timestamp}.npy",   k_np)
np.save(OUT_DIR / f"lam_test_{timestamp}.npy", lam_np)

with open(OUT_DIR / f"results_{timestamp}.json", 'w', encoding='utf-8') as f:
    json.dump({'transition':'P_to_N',
               'metrics': checkpoint['metrics'],
               'weibull_params': checkpoint['weibull_params']}, f, indent=2)

print(f"\n✅ Modelo P→N congelado: {pth.name}")
print(f"\n🎯 Próximo passo: modelo de riscos competitivos (P→S + P→N)")